In [0]:
from pyspark.sql.functions import *

silver_df = spark.table("workspace.default.silver_pet_store_records")

display(silver_df)

In [0]:
from pyspark.sql.functions import *

new_data = [
    (9991, "Medicine", 150, 12000, 1, "VC_999", "USA", "small", "dog", 9, 1),
    (9992, "Supplements", 90, 8500, 1, "VC_998", "INDIA", "medium", "cat", 8, 1),
    (9993, "Toys", 45, 2500, 0, "VC_997", "USA", "small", "rabbit", 7, 0)
]

columns = [
    "product_id", "product_category", "sales", "price", "VAP",
    "vendor_id", "country", "pet_size", "pet_type", "rating", "re_buy"
]

incremental_df = spark.createDataFrame(new_data, columns)

incremental_df = (
    incremental_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("incremental_daily_file"))
    .withColumn("pipeline_layer", lit("silver"))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("estimated_revenue", col("sales") * col("price"))
)

display(incremental_df)

In [0]:
incremental_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.incremental_pet_store_records")

In [0]:
spark.sql("""
MERGE INTO workspace.default.silver_pet_store_records AS target
USING workspace.default.incremental_pet_store_records AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
result_df = spark.sql("""
SELECT *
FROM workspace.default.silver_pet_store_records
WHERE product_id IN (9991, 9992, 9993)
""")

display(result_df)

In [0]:
print("Updated Silver Row Count:", spark.table("workspace.default.silver_pet_store_records").count())